In [1]:
import sys
import pathlib
import pandas as pd

In [2]:
import warnings
warnings.simplefilter('ignore', FutureWarning)

import pandas as pd

In [3]:
_p = pathlib.Path.cwd()

while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent

if not (_p / "db" / "dataraflow_db.py").exists():
    raise FileNotFoundError(
        "Could not find db/dataraflow_db.py. "
        "Make sure VS Code is using your DataraFlow project."
    )

sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query

print("Database connection helper imported successfully")

Database connection helper imported successfully


In [4]:
gdp=run_query("""
  SELECT country,[NY.GDP.MKTP.CD] AS gdp
  FROM week08.wb_gdp_2013
""")
gdp.head()

,country,gdp
0,Arab World,2.827557e+12
1,Caribbean small states,6.928705e+10
2,Central Europe and the Baltics,1.419384e+12
3,East Asia & Pacific (all income levels),2.084673e+13
4,East Asia & Pacific (developing only),1.172940e+13


In [5]:
cols = run_query("""
SELECT TOP 5 * 
FROM week08.wb_gdp_2013
""")

cols.head()
print(cols.columns.tolist())

['country', 'year', 'NY.GDP.MKTP.CD']


In [6]:
tables = run_query("""
SELECT TABLE_NAME 
FROM INFORMATION_SCHEMA.TABLES 
WHERE TABLE_SCHEMA = 'week08'
""")
tables

,TABLE_NAME
0,wb_gdp_2013
1,wb_le_2013
2,wb_pop_2013


In [7]:
le = run_query("""
  SELECT country,[SP.DYN.LE00.IN] AS life_expectancy
  FROM week08.wb_le_2013
""")
le.head()

,country,life_expectancy
0,Arab World,70.631305
1,Caribbean small states,71.901964
2,Central Europe and the Baltics,76.127583
3,East Asia & Pacific (all income levels),74.604619
4,East Asia & Pacific (developing only),73.657617


In [8]:
pop = run_query("""
  SELECT country,[SP.POP.TOTL] AS population
  FROM week08.wb_pop_2013
""")
pop.head()

,country,population
0,Arab World,3.770967e+08
1,Caribbean small states,6.975819e+06
2,Central Europe and the Baltics,1.037137e+08
3,East Asia & Pacific (all income levels),2.248867e+09
4,East Asia & Pacific (developing only),2.006073e+09


In [9]:
print("gdp dataset shape:", gdp.shape)
print("le dataset shape:", le.shape)
print("pop dataset shape:", pop.shape)

gdp dataset shape: (248, 2)
le dataset shape: (248, 2)
pop dataset shape: (248, 2)


In [10]:
combined=gdp.merge(le,on="country")
combined.head()

,country,gdp,life_expectancy
0,Arab World,2.827557e+12,70.631305
1,Caribbean small states,6.928705e+10,71.901964
2,Central Europe and the Baltics,1.419384e+12,76.127583
3,East Asia & Pacific (all income levels),2.084673e+13,74.604619
4,East Asia & Pacific (developing only),1.172940e+13,73.657617


In [11]:
combined=combined.merge(pop,on="country")
combined.head()

,country,gdp,life_expectancy,population
0,Arab World,2.827557e+12,70.631305,3.770967e+08
1,Caribbean small states,6.928705e+10,71.901964,6.975819e+06
2,Central Europe and the Baltics,1.419384e+12,76.127583,1.037137e+08
3,East Asia & Pacific (all income levels),2.084673e+13,74.604619,2.248867e+09
4,East Asia & Pacific (developing only),1.172940e+13,73.657617,2.006073e+09


In [12]:
print("combined dataset shape")
print("column names:", combined.columns.tolist())
combined.head(10)

combined dataset shape
column names: ['country', 'gdp', 'life_expectancy', 'population']


,country,gdp,life_expectancy,population
0,Arab World,2.827557e+12,70.631305,3.770967e+08
1,Caribbean small states,6.928705e+10,71.901964,6.975819e+06
2,Central Europe and the Baltics,1.419384e+12,76.127583,1.037137e+08
3,East Asia & Pacific (all income levels),2.084673e+13,74.604619,2.248867e+09
4,East Asia & Pacific (developing only),1.172940e+13,73.657617,2.006073e+09
5,Euro area,1.318878e+13,81.373197,3.372839e+08
6,Europe & Central Asia (all income levels),2.294990e+13,76.789732,8.988371e+08
7,Europe & Central Asia (developing only),1.879645e+12,72.456461,2.624163e+08
8,European Union,1.798747e+13,80.384262,5.065580e+08
9,Fragile and conflict affected situations,7.315581e+11,61.771105,4.617132e+08


In [13]:
combined = run_query("""
    SELECT g.country,
           g.[NY.GDP.MKTP.CD] AS gdp,
           l.[SP.DYN.LE00.IN] AS life_expectancy,
           p.[SP.POP.TOTL]    AS population
    FROM week08.wb_gdp_2013 g
    JOIN week08.wb_le_2013  l ON g.country = l.country
    JOIN week08.wb_pop_2013 p ON g.country = p.country
""")
combined.shape   # (248, 4)

(248, 4)

In [14]:
combined = run_query("""
    SELECT g.country,
           g.[NY.GDP.MKTP.CD] AS gdp,
           l.[SP.DYN.LE00.IN] AS life_expectancy,
           p.[SP.POP.TOTL]    AS population
    FROM week08.wb_gdp_2013 g
    JOIN week08.wb_le_2013  l ON g.country = l.country
    JOIN week08.wb_pop_2013 p ON g.country = p.country
""")
combined.head()

,country,gdp,life_expectancy,population
0,Afghanistan,2.045894e+10,60.028268,30682500.0
1,Albania,1.278103e+10,77.537244,2897366.0
2,Algeria,2.097035e+11,74.568951,38186135.0
3,American Samoa,NaN,NaN,55302.0
4,Andorra,3.249101e+09,NaN,75902.0


In [15]:
top_life = run_query("""
    SELECT TOP 20
        g.country,
        g.[NY.GDP.MKTP.CD] AS gdp,
        l.[SP.DYN.LE00.IN] AS life_expectancy
    FROM week08.wb_gdp_2013 AS g
    JOIN week08.wb_le_2013 AS l
        ON g.country = l.country
    ORDER BY g.[NY.GDP.MKTP.CD] DESC
""")

top_life

,country,gdp,life_expectancy
0,World,7.623680e+13,71.209615
1,High income,5.225619e+13,79.109783
2,OECD members,4.800976e+13,79.921974
3,High income: OECD,4.592731e+13,80.650435
4,Low & middle income,2.401149e+13,69.311130
5,Middle income,2.363712e+13,70.304864
6,Europe & Central Asia (all income levels),2.294990e+13,76.789732
7,East Asia & Pacific (all income levels),2.084673e+13,74.604619
8,North America,1.861259e+13,79.097649
9,Upper middle income,1.815501e+13,74.317090


In [16]:
average_life_expectancy= top_life["life_expectancy"].mean()
print("Average_life_expectancy", round(average_life_expectancy,2))


Average_life_expectancy 75.92


In [17]:
#TRANSFORMATION#
table = [
    ["UK", 2678454886796.7],
    ["USA", 16768100000000.0],
    ["China", 9240270452047.0],
    ["Brazil", 2245673032353.8],
    ["South Africa", 366057913367.1]
]

headings = ["Country", "GDP (US$)"]

gdp = pd.DataFrame(columns=headings, data=table)

gdp

,Country,GDP (US$)
0,UK,2.678455e+12
1,USA,1.676810e+13
2,China,9.240270e+12
3,Brazil,2.245673e+12
4,South Africa,3.660579e+11


In [18]:
headings = ["Country name", "Life expectancy (years)"]

table = [
    ["China", 75],
    ["Russia", 71],
    ["United States", 79],
    ["India", 66],
    ["United Kingdom", 81]
]

life = pd.DataFrame(columns=headings, data=table)

life

,Country name,Life expectancy (years)
0,China,75
1,Russia,71
2,United States,79
3,India,66
4,United Kingdom,81


In [19]:
def expandCountry(name):
    if name == "UK":
        return "United Kingdom"
    elif name == "USA":
        return "United States"
    else:
        return name

In [20]:
print(expandCountry("UK"))
print(expandCountry("USA"))
print(expandCountry("China"))

United Kingdom
United States
China


In [21]:
gdp["Country name"]=gdp["Country"].apply(expandCountry)

In [22]:
def usdToGBP(usd): return usd/1.564768

In [23]:
def roundToMillions(value): return round(value/1000000)

In [ ]:
gdp["GDP (£m)"] = (
    gdp["GDP (US$)"]
    .apply(usdToGBP)
    .apply(roundToMillions)
)

gdp

,Country,GDP (US$),Country name,GDP (£m)
0,UK,2.678455e+12,United Kingdom,1711727
1,USA,1.676810e+13,United States,10716029
2,China,9.240270e+12,China,5905202
3,Brazil,2.245673e+12,Brazil,1435148
4,South Africa,3.660579e+11,South Africa,233937


In [25]:
headings=["Country name", "GDP (£m)"]
print(headings)

['Country name', 'GDP (£m)']


In [26]:
gdp_original=gdp.copy()

In [27]:
pd.merge(gdp, life, on="Country name", how="left")

,Country,GDP (US$),Country name,GDP (£m),Life expectancy (years)
0,UK,2.678455e+12,United Kingdom,1711727,81.0
1,USA,1.676810e+13,United States,10716029,79.0
2,China,9.240270e+12,China,5905202,75.0
3,Brazil,2.245673e+12,Brazil,1435148,NaN
4,South Africa,3.660579e+11,South Africa,233937,NaN


In [28]:
pd.merge(gdp, life, on="Country name", how="right")

,Country,GDP (US$),Country name,GDP (£m),Life expectancy (years)
0,China,9.240270e+12,China,5905202.0,75
1,NaN,NaN,Russia,NaN,71
2,USA,1.676810e+13,United States,10716029.0,79
3,NaN,NaN,India,NaN,66
4,UK,2.678455e+12,United Kingdom,1711727.0,81


In [29]:
pd.merge(gdp, life, on="Country name", how="outer")

,Country,GDP (US$),Country name,GDP (£m),Life expectancy (years)
0,Brazil,2.245673e+12,Brazil,1435148.0,NaN
1,China,9.240270e+12,China,5905202.0,75.0
2,NaN,NaN,India,NaN,66.0
3,NaN,NaN,Russia,NaN,71.0
4,South Africa,3.660579e+11,South Africa,233937.0,NaN
5,UK,2.678455e+12,United Kingdom,1711727.0,81.0
6,USA,1.676810e+13,United States,10716029.0,79.0


In [30]:
import pandas as pd

population_df = pd.DataFrame({
    'Country name': ['Nigeria', 'Ghana', 'Kenya', 'Egypt'],
    'Life expectancy (years)': [54, 64, 66, 72]
})

population_df

,Country name,Life expectancy (years)
0,Nigeria,54
1,Ghana,64
2,Kenya,66
3,Egypt,72


In [31]:

gdpVsLife=pd.merge(gdp,life,on='Country name', how='inner')

In [32]:
# Use gdpVsLife as the second dataframe, not 'Country name'
join_inner = pd.merge(population_df, gdpVsLife, on='Country name', how='inner')
join_left = pd.merge(population_df, gdpVsLife, on='Country name', how='left')
join_right = pd.merge(population_df, gdpVsLife, on='Country name', how='right')
join_outer = pd.merge(population_df, gdpVsLife, on='Country name', how='outer')

print("inner:", join_inner.shape)
print("left:", join_left.shape)
print("right:", join_right.shape)
print("outer:", join_outer.shape)

join_outer

inner: (0, 6)
left: (4, 6)
right: (3, 6)
outer: (7, 6)


,Country name,Life expectancy (years)_x,Country,GDP (US$),GDP (£m),Life expectancy (years)_y
0,China,NaN,China,9.240270e+12,5905202.0,75.0
1,Egypt,72.0,NaN,NaN,NaN,NaN
2,Ghana,64.0,NaN,NaN,NaN,NaN
3,Kenya,66.0,NaN,NaN,NaN,NaN
4,Nigeria,54.0,NaN,NaN,NaN,NaN
5,United Kingdom,NaN,UK,2.678455e+12,1711727.0,81.0
6,United States,NaN,USA,1.676810e+13,10716029.0,79.0


In [33]:
print("left:", join_left)


left:   Country name  Life expectancy (years)_x Country  GDP (US$)  GDP (£m)  \
0      Nigeria                         54     NaN        NaN       NaN   
1        Ghana                         64     NaN        NaN       NaN   
2        Kenya                         66     NaN        NaN       NaN   
3        Egypt                         72     NaN        NaN       NaN   

   Life expectancy (years)_y  
0                        NaN  
1                        NaN  
2                        NaN  
3                        NaN  


In [34]:
print("right:", join_right)

right:      Country name  Life expectancy (years)_x Country     GDP (US$)  GDP (£m)  \
0  United Kingdom                        NaN      UK  2.678455e+12   1711727   
1   United States                        NaN     USA  1.676810e+13  10716029   
2           China                        NaN   China  9.240270e+12   5905202   

   Life expectancy (years)_y  
0                         81  
1                         79  
2                         75  


In [35]:
print("inner:", join_inner)

inner: Empty DataFrame
Columns: [Country name, Life expectancy (years)_x, Country, GDP (US$), GDP (£m), Life expectancy (years)_y]
Index: []


In [36]:
print("outer:", join_outer)

outer:      Country name  Life expectancy (years)_x Country     GDP (US$)  \
0           China                        NaN   China  9.240270e+12   
1           Egypt                       72.0     NaN           NaN   
2           Ghana                       64.0     NaN           NaN   
3           Kenya                       66.0     NaN           NaN   
4         Nigeria                       54.0     NaN           NaN   
5  United Kingdom                        NaN      UK  2.678455e+12   
6   United States                        NaN     USA  1.676810e+13   

     GDP (£m)  Life expectancy (years)_y  
0   5905202.0                       75.0  
1         NaN                        NaN  
2         NaN                        NaN  
3         NaN                        NaN  
4         NaN                        NaN  
5   1711727.0                       81.0  
6  10716029.0                       79.0  


In [37]:
#TASK,POPULATION DATAFRAME (Country name, Life expectancy (years))#
population_df=pd.DataFrame({

    'Country name': ['Nigeria', 'Kenya', 'Libya'],
    'Life expectancy (years)': [54,64,74]
})

In [38]:
combined.to_csv("wb_combined_2013new.csv", index=False)
print("Combined World Bank dataset saved successfully")

Combined World Bank dataset saved successfully
